In [2]:
import requests, feedparser, json,time, os, xmltodict
import polars as pl
from config_info import APIS
from pprint import pprint

from scrapers.arxiv import parser_arxiv
from scrapers.hal import parser_hal

# Basic Fetch

In [3]:
def fetch_raw(url, headers=None):
    headers = headers or {"User-Agent":  "IntelliCorpus/1.0 (contact: paull@scholar-cergy.com)"}
    r = requests.get(url, headers=headers, timeout=15)
    r.raise_for_status()
    content_type = r.headers.get("Content-Type","")
    if 'xml' in content_type:
        return r.text
    elif 'json' in content_type:
        return r.json()
    else:
        return r.text

# Semantic Scholar

In [4]:
def semantic_fetch(results):
    for result in results["data"]:
        print(result)
        # query = APIS["Semantic Scholar"]["paper_url"].format(paper_id=result["paperId"])
        # res = fetch_raw(query)
        # print(res)
        # print(query)

# CORE

In [4]:
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.getenv("CORE_API_KEY")
# url = "https://api.core.ac.uk/v3/search/works"

# headers = {
#     "Authorization": f"Bearer {API_KEY}"
# }

# params = { 
#     "q": "AI agent",
#     "limit": 5
# }

# r = requests.get(url, headers=headers, params=params)
# r.raise_for_status()

# data = r.json()
# # print(data.keys())
# pprint(data)


# Main 

In [5]:
# raw_result = fetch_raw(APIS["arXiv"]["api_url"].format(query="AI agent",quantity='1'))
# result_arxiv =  parser_arxiv(raw_result)
# print(type(result_arxiv)) # Arxiv Ok
 
# Test HAL
# url_hal = fetch_raw(APIS["HAL"]["api_url"].format(query="AI agent",quantity='2'))
# result_hal = parser_hal(url_hal)
# pprint(result_hal[0]) #Good 

# Test PubMed
from scrapers.pubmed import fetch_pubmed
xml_batches_pubmed = fetch_pubmed(
    query="AI agent",
    max_results=1,
    batch_size=1,
    email="proliquer@scholar-perigueuxu.com"
)
pprint(xml_batches_pubmed)

# Test Sementic Scholar
# url_sem = fetch_raw(APIS["Semantic Scholar"]["api_url"].format(query="AI agent",quantity='5'))
# print(type(url_sem))
# semantic_fetch(url_sem)


['<?xml version="1.0" ?>\n'
 '<!DOCTYPE PubmedArticleSet PUBLIC "-//NLM//DTD PubMedArticle, 1st January '
 '2025//EN" "https://dtd.nlm.nih.gov/ncbi/pubmed/out/pubmed_250101.dtd">\n'
 '<PubmedArticleSet>\n'
 '<PubmedArticle><MedlineCitation Status="MEDLINE" Owner="NLM" '
 'IndexingMethod="Automated"><PMID '
 'Version="1">41860380</PMID><DateCompleted><Year>2026</Year><Month>03</Month><Day>20</Day></DateCompleted><DateRevised><Year>2026</Year><Month>03</Month><Day>20</Day></DateRevised><Article '
 'PubModel="Print"><Journal><ISSN '
 'IssnType="Electronic">1089-7682</ISSN><JournalIssue '
 'CitedMedium="Internet"><Volume>36</Volume><Issue>3</Issue><PubDate><Year>2026</Year><Month>Mar</Month><Day>01</Day></PubDate></JournalIssue><Title>Chaos '
 '(Woodbury, '
 'N.Y.)</Title><ISOAbbreviation>Chaos</ISOAbbreviation></Journal><ArticleTitle>Mutation '
 'promote cooperation in repeated games on structured '
 'populations.</ArticleTitle><ELocationID EIdType="pii" '
 'ValidYN="Y">033138</ELocationI

In [30]:
from scrapers.pubmed import format_hal_data
# pprint(len(xml_batches_pubmed))
for result in xml_batches_pubmed:
        parsed_dict = xmltodict.parse(result, dict_constructor=dict)
        articles = parsed_dict.get("PubmedArticleSet", {}).get("PubmedArticle", [])
        if isinstance(articles, dict):
                articles = [articles]
        for article in articles:
                citation = article.get("MedlineCitation", {})
                article_info = citation.get("Article", {})
                pprint(citation.keys())
                pprint(citation)
                pmid = citation.get("PMID", {}).get("#text")
                titre = article_info.get("ArticleTitle")
                summary = article_info.get("Abstract", {}).get("AbstractText") 
                published_at = citation.get("DateCompleted")
        # citations = parsed_dict.get("MedlineCitation",{})
        # print(citations)
        # pprint(parsed_dict)

dict_keys(['@Status', '@Owner', '@IndexingMethod', 'PMID', 'DateCompleted', 'DateRevised', 'Article', 'MedlineJournalInfo', 'CitationSubset', 'MeshHeadingList'])
{'@IndexingMethod': 'Automated',
 '@Owner': 'NLM',
 '@Status': 'MEDLINE',
 'Article': {'@PubModel': 'Print',
             'Abstract': {'AbstractText': 'Cooperation is a fundamental '
                                          'phenomenon in human societies and '
                                          'multi-agent systems, yet it remains '
                                          'challenging to sustain among '
                                          'rational individuals. In realistic '
                                          'evolutionary processes, cooperative '
                                          'behavior is inevitably affected by '
                                          'behavioral variation and random '
                                          'perturbations; a common way to '
                           

In [ ]:
from IPython.display import JSON    
def parse_xml_to_json_and_display(xml_string: str) -> dict:
    """
    Convertit une chaîne XML en dictionnaire Python (JSON) et l'affiche joliment.
    """
    if not xml_string:
        print("Le XML fourni est vide.")
        return {}

    # 1. Conversion du XML en Dictionnaire Python
    # dict_constructor=dict permet d'avoir des dictionnaires standards
    parsed_dict = xmltodict.parse(xml_string, dict_constructor=dict)
    
    # 2. Le "Beau Rendu" (Pretty Print)
    # indent=4 crée de belles indentations pour lire facilement la structure
    # ensure_ascii=False permet de bien afficher les accents français (é, à, etc.)
    json_formate = json.dumps(parsed_dict, indent=4, ensure_ascii=False)
    
    # print("--- 🌟 Aperçu des données extraites ---")
    # print(json_formate)
    # print("--------------------------------------")
    # 3. On retourne le dictionnaire pour la suite du pipeline
    return parsed_dict

data_pubmed = parse_xml_to_json_and_display(xml_batches_pubmed[0])
# def save_data_to_file(data_dict: dict, filename: str = "retour_api.json"):
#     """
#     Sauvegarde l'intégralité d'un dictionnaire dans un fichier JSON local.
#     """
#     # On ouvre un fichier en mode écriture ("w") avec le bon encodage
#     with open(filename, "w", encoding="utf-8") as file:
#         json.dump(data_dict, file, indent=4, ensure_ascii=False)
        
#     print(f"✅ L'intégralité des données a été sauvegardée dans le fichier : '{filename}'")

# save_data_to_file(d, "resultat_hal.json")

In [ ]:
from database.postgres.crud import upsert_data
from models.postgres.corpus_schema import document_table 
from config.db_engine import get_db_engine
from processing.cleaning_data import normalize_data

arxiv_mapping = {
    "published_at": "published",
}

# clean_arxiv_data = normalize_data(
#     raw_data=result_arxiv, 
#     source_name="arXiv",
#     date_columns=["published_at"],
#     columns_drop=["updated"]
#     )

# hal_mapping = { "published" : "published_at"}
# clean_hal_data = normalize_data(
#     raw_data=result_hal,
#     source_name="Hal",
#     column_mapping=hal_mapping,
#     date_columns=["published"]
# )
engine = get_db_engine()
# upsert_data(clean_hal_data, ['id'], document_table, engine)
